# NB20 — Sklearn Pipelines: Production-Ready Workflow

> **StatQuest: "All your preprocessing steps must be INSIDE the pipeline — otherwise you're cheating (data leakage)."**

---

## The main ideas:

1. **Data leakage:** fitting the scaler on ALL data before CV means test-fold info leaks into training -> CV score is optimistic
2. **Pipeline:** chains preprocessing + model into ONE object -> CV is applied correctly
3. **GridSearchCV on a pipeline:** tunes ALL steps together, prevents leakage automatically
4. **joblib:** serialise and load the pipeline for production serving
5. One `predict()` call applies all preprocessing + model automatically


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Raw data\n(missing values,\nunscaled)',
        'Pipeline step 1:\nImputer\n(fill missing)',
        'Pipeline step 2:\nStandardScaler\n(mean=0, std=1)',
        'Pipeline step 3:\nRegression\nmodel',
        'GridSearchCV\non whole\npipeline',
        'joblib.dump\n-> .pkl file\n(deployment)',
        'joblib.load\n-> predict()\n(one call)',
    ],
    title='NB20 Conceptual Flow: sklearn Pipeline for Production ML',
    colors=['#37474F','#1565C0','#2E7D32','#E65100','#6A1B9A','#C62828','#00695C'],
    figsize=(16, 2.8),
)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.datasets import fetch_california_housing
import warnings; warnings.filterwarnings('ignore')

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# Inject 3% missing values
np.random.seed(42)
mask = np.random.rand(*X.shape) < 0.03
X_miss = X.copy(); X_miss[mask] = np.nan

X_tr, X_te, y_tr, y_te = train_test_split(X_miss, y, test_size=0.2, random_state=0)

print(f"Missing values in train: {X_tr.isna().sum().sum()}")
print(f"Missing values in test:  {X_te.isna().sum().sum()}")
print()
print("--- WRONG way (data leakage) ---")
# Scaler fitted on ALL data before split -> leaks test set info
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_scaled_all = sc.fit_transform(X_miss.fillna(X_miss.median()))
X_tr_wrong, X_te_wrong, y_tr_wrong, y_te_wrong = train_test_split(X_scaled_all, y, test_size=0.2, random_state=0)
m_wrong = Ridge(alpha=1).fit(X_tr_wrong, y_tr_wrong)
print(f"  CV R^2 (inflated): {m_wrong.score(X_tr_wrong, y_tr_wrong):.4f}")

print()
print("--- RIGHT way (Pipeline) ---")
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=1.0)),
])
pipe.fit(X_tr, y_tr)
print(f"  Test R^2: {pipe.score(X_te, y_te):.4f}  <- honest estimate")


In [ ]:
# GridSearchCV on the whole pipeline — no leakage
import numpy as np
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__alpha': np.logspace(-2, 3, 15),  # prefix = step_name + __
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_tr, y_tr)

print(f"Best alpha:      {grid.best_params_['model__alpha']:.4f}")
print(f"Best 5-fold CV R^2: {grid.best_score_:.4f}")
print(f"Test R^2:           {grid.score(X_te, y_te):.4f}")

import pandas as pd
cv_df = pd.DataFrame(grid.cv_results_)[['param_model__alpha','mean_test_score','std_test_score']]
cv_df.columns = ['alpha','mean_R2','std_R2']
print("\nAll alpha values tested:")
print(cv_df.round(4).to_string(index=False))


In [ ]:
# Save and load pipeline for production
import joblib, os, pandas as pd, numpy as np

save_path = os.path.join(r'C:/Users/Administrator/ML/Linear Regression', 'best_pipeline.pkl')
joblib.dump(grid.best_estimator_, save_path)
print(f"Saved to: {save_path}")

loaded = joblib.load(save_path)
y_pred = loaded.predict(X_te)
from sklearn.metrics import r2_score, mean_squared_error
print(f"Loaded pipeline Test R^2:  {r2_score(y_te, y_pred):.4f}")
print(f"Loaded pipeline Test RMSE: {np.sqrt(mean_squared_error(y_te, y_pred)):.4f}")

# Predict a single new observation (with missing value — handled automatically)
new_obs = pd.DataFrame([X.median().values], columns=X.columns)
new_obs.iloc[0, 0] = np.nan
pred = loaded.predict(new_obs)
print(f"\nPrediction for median house: ${pred[0]*100:.0f}k")
print("(Imputer and scaler applied automatically inside predict())")


## Complete curriculum summary

| NB | Topic | Core concept (one line) |
|----|-------|------------------------|
| 01 | Intuition | OLS = find the line that minimises sum of squared residuals |
| 02 | Derivation | b1 = Cov(x,y)/Var(x) comes from setting dSSR/db=0 |
| 03 | From scratch | Implement every formula in pure Python |
| 04 | R^2 | R^2 = SS_reg/SS_tot = fraction of variance explained |
| 05 | Inference | t = b/SE; p-value = probability of this t if H0 true |
| 06 | Assumptions | LINE: Linearity, Independence, Normality, Equal variance |
| 07 | Diagnostics | 4 plots: residuals vs fitted, Q-Q, scale-location, leverage |
| 08 | Multiple | b = (XtX)^-1 Xty; each b_j is ceteris paribus |
| 09 | Collinearity | VIF > 10 = severe; Ridge is the fix |
| 10 | Poly/interact | Add x^2 as feature; x1*x2 for "it depends" effects |
| 11 | Ridge (L2) | SSR + lambda*sum(b^2); shrinks but never zeroes |
| 12 | Lasso (L1) | SSR + lambda*sum(|b|); diamond corners -> exact zeros |
| 13 | ElasticNet | L1+L2; sparsity + stability for correlated features |
| 14 | Cross-val | k-fold CV; learning curves for bias/variance diagnosis |
| 15 | GD | Take steps downhill; foundation of deep learning |
| 16 | Robust | Huber/RANSAC/Theil-Sen resist outliers |
| 17 | Quantile | Model median or any percentile; natural prediction bands |
| 18 | Bayesian | Prior + likelihood = posterior distribution over b |
| 19 | Time series | Add t, sin/cos, lag features for temporal structure |
| 20 | Pipelines | Chain imputer + scaler + model; no leakage; one predict() |
